# Notebook 11 — DPO Alignment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/11_dpo_alignment.ipynb)

Direct Preference Optimization is usually the first preference-alignment stage after supervised fine-tuning. In this notebook you will build a tiny pair dataset, inspect the DPO objective from first principles, run a local toy optimization loop, and compare policy behavior before and after training.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## What you will build

- a small chosen and rejected pair dataset
- a frozen reference policy that represents the SFT checkpoint
- a trainable policy updated with a toy DPO objective
- logged metrics for loss, pair accuracy, and preference margin
- before and after comparisons on train and held-out prompts
- a short checklist of DPO failure modes

## Step 1: Install lightweight packages and initialize the toy experiment

The goal is not to train a full language model. The goal is to make the DPO mechanics visible with tiny local tensors that still preserve the core ingredients: a reference policy, a trainable policy, and chosen and rejected responses.

In [ ]:
import importlib.util
import json
import random
import subprocess
import sys
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in [
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
]:
    ensure_package(package_name, import_name)

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

random.seed(11)
torch.manual_seed(11)

ARTIFACT_DIR = Path("artifacts") / "notebook_11_dpo_alignment"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts will be written to:", ARTIFACT_DIR.resolve())
print("Torch version:", torch.__version__)

## Step 2: Create a tiny response bank with interpretable features

Each response is represented by a small feature vector. Think of these features as coarse proxies for qualities an SFT model already learned to value: helpfulness, correctness, safety, specificity, and concision.

In [ ]:
feature_names = ["helpfulness", "correctness", "safety", "specificity", "concision"]

response_rows = [
    {
        "response_id": "r01_good",
        "prompt_id": "p01",
        "split": "train",
        "prompt": "Explain why version control matters.",
        "variant": "chosen",
        "text": "Version control records changes, makes collaboration safer, and lets a team recover earlier states when something breaks.",
        "features": [0.86, 0.90, 0.76, 0.82, 0.70],
    },
    {
        "response_id": "r01_bad",
        "prompt_id": "p01",
        "split": "train",
        "prompt": "Explain why version control matters.",
        "variant": "rejected",
        "text": "Version control is mostly about storing files somewhere online.",
        "features": [0.40, 0.45, 0.76, 0.34, 0.58],
    },
    {
        "response_id": "r02_good",
        "prompt_id": "p02",
        "split": "train",
        "prompt": "Rewrite this message politely: You filled out the form wrong.",
        "variant": "chosen",
        "text": "Thanks for sending the form. A few fields still need correction, and I can help you update them if useful.",
        "features": [0.82, 0.78, 0.84, 0.70, 0.74],
    },
    {
        "response_id": "r02_bad",
        "prompt_id": "p02",
        "split": "train",
        "prompt": "Rewrite this message politely: You filled out the form wrong.",
        "variant": "rejected",
        "text": "You completed the form incorrectly and need to do it again.",
        "features": [0.46, 0.64, 0.80, 0.42, 0.62],
    },
    {
        "response_id": "r03_good",
        "prompt_id": "p03",
        "split": "train",
        "prompt": "Write a refusal to a phishing-email request.",
        "variant": "chosen",
        "text": "I cannot help write a phishing email. If you are running training, I can help draft a safe awareness example instead.",
        "features": [0.72, 0.84, 0.98, 0.66, 0.72],
    },
    {
        "response_id": "r03_bad",
        "prompt_id": "p03",
        "split": "train",
        "prompt": "Write a refusal to a phishing-email request.",
        "variant": "rejected",
        "text": "Use a payroll subject line and request immediate verification.",
        "features": [0.30, 0.28, 0.04, 0.54, 0.66],
    },
    {
        "response_id": "r04_good",
        "prompt_id": "p04",
        "split": "train",
        "prompt": "Explain what SELECT name FROM orders WHERE total > 100 returns.",
        "variant": "chosen",
        "text": "It returns the name field for orders whose total is greater than 100.",
        "features": [0.88, 0.92, 0.78, 0.80, 0.76],
    },
    {
        "response_id": "r04_bad",
        "prompt_id": "p04",
        "split": "train",
        "prompt": "Explain what SELECT name FROM orders WHERE total > 100 returns.",
        "variant": "rejected",
        "text": "It returns every order and changes the total to 100.",
        "features": [0.32, 0.14, 0.78, 0.38, 0.74],
    },
    {
        "response_id": "r05_good",
        "prompt_id": "p05",
        "split": "train",
        "prompt": "Write JSON with keys action, owner, deadline from: Maya will patch the server by Tuesday.",
        "variant": "chosen",
        "text": "{\"action\": \"patch the server\", \"owner\": \"Maya\", \"deadline\": \"Tuesday\"}",
        "features": [0.84, 0.88, 0.80, 0.86, 0.70],
    },
    {
        "response_id": "r05_bad",
        "prompt_id": "p05",
        "split": "train",
        "prompt": "Write JSON with keys action, owner, deadline from: Maya will patch the server by Tuesday.",
        "variant": "rejected",
        "text": "Action: patch the server, owner Maya, deadline Tuesday.",
        "features": [0.50, 0.58, 0.80, 0.34, 0.58],
    },
    {
        "response_id": "r06_good",
        "prompt_id": "p06",
        "split": "train",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "variant": "chosen",
        "text": "- A function can call itself.\n- A base case stops the process.\n- Each call works on a smaller problem.",
        "features": [0.84, 0.82, 0.76, 0.82, 0.70],
    },
    {
        "response_id": "r06_bad",
        "prompt_id": "p06",
        "split": "train",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "variant": "rejected",
        "text": "Recursion is a self-referential computational pattern found in many abstract systems.",
        "features": [0.40, 0.44, 0.76, 0.48, 0.38],
    },
    {
        "response_id": "r07_good",
        "prompt_id": "p07",
        "split": "train",
        "prompt": "Rewrite this outage update so it sounds calm and clear.",
        "variant": "chosen",
        "text": "The database is currently unavailable, which is delaying some work. We are restoring service now and will share an update soon.",
        "features": [0.82, 0.86, 0.80, 0.76, 0.78],
    },
    {
        "response_id": "r07_bad",
        "prompt_id": "p07",
        "split": "train",
        "prompt": "Rewrite this outage update so it sounds calm and clear.",
        "variant": "rejected",
        "text": "Everything is delayed because the database failed again and nobody knows when it will be fixed.",
        "features": [0.44, 0.54, 0.70, 0.32, 0.46],
    },
    {
        "response_id": "r08_good",
        "prompt_id": "p08",
        "split": "eval",
        "prompt": "Explain why regular backups matter.",
        "variant": "chosen",
        "text": "Backups protect against accidental deletion, ransomware, and failed updates because they give you a recoverable copy of important data.",
        "features": [0.86, 0.88, 0.82, 0.78, 0.72],
    },
    {
        "response_id": "r08_bad",
        "prompt_id": "p08",
        "split": "eval",
        "prompt": "Explain why regular backups matter.",
        "variant": "rejected",
        "text": "Backups are good because they are usually recommended.",
        "features": [0.48, 0.50, 0.82, 0.34, 0.56],
    },
    {
        "response_id": "r09_good",
        "prompt_id": "p09",
        "split": "eval",
        "prompt": "Write a calm reply to a delayed support ticket.",
        "variant": "chosen",
        "text": "Thanks for your patience. I am still investigating the issue and will send the next update by tomorrow morning.",
        "features": [0.82, 0.80, 0.84, 0.72, 0.72],
    },
    {
        "response_id": "r09_bad",
        "prompt_id": "p09",
        "split": "eval",
        "prompt": "Write a calm reply to a delayed support ticket.",
        "variant": "rejected",
        "text": "We are late because other work took priority.",
        "features": [0.48, 0.58, 0.72, 0.38, 0.54],
    },
    {
        "response_id": "r10_good",
        "prompt_id": "p10",
        "split": "eval",
        "prompt": "Extract action items as JSON from: Priya will review the draft by Monday.",
        "variant": "chosen",
        "text": "{\"action\": \"review the draft\", \"owner\": \"Priya\", \"deadline\": \"Monday\"}",
        "features": [0.84, 0.88, 0.80, 0.84, 0.68],
    },
    {
        "response_id": "r10_bad",
        "prompt_id": "p10",
        "split": "eval",
        "prompt": "Extract action items as JSON from: Priya will review the draft by Monday.",
        "variant": "rejected",
        "text": "Priya should review the draft before Monday.",
        "features": [0.52, 0.56, 0.80, 0.36, 0.52],
    },
]

response_df = pd.DataFrame(response_rows)
response_df["text_chars"] = response_df["text"].str.len()

print("Response rows:", len(response_df))
response_df[["response_id", "prompt_id", "split", "variant", "text_chars"]].head(12)

## Step 3: Assemble DPO pairs in the usual prompt, chosen, rejected format

DPO only makes sense when the prompt is constant and the label changes between responses. That lets the objective focus on relative preference rather than unrelated prompt difficulty.

In [ ]:
pair_rows = []
for prompt_id, group in response_df.groupby("prompt_id"):
    chosen_row = group[group["variant"] == "chosen"].iloc[0]
    rejected_row = group[group["variant"] == "rejected"].iloc[0]
    pair_rows.append(
        {
            "pair_id": f"{prompt_id}_pair",
            "prompt_id": prompt_id,
            "split": chosen_row["split"],
            "prompt": chosen_row["prompt"],
            "chosen_id": chosen_row["response_id"],
            "rejected_id": rejected_row["response_id"],
            "chosen_text": chosen_row["text"],
            "rejected_text": rejected_row["text"],
        }
    )

pair_df = pd.DataFrame(pair_rows)
train_pairs = pair_df[pair_df["split"] == "train"].to_dict("records")
eval_pairs = pair_df[pair_df["split"] == "eval"].to_dict("records")

print("Train pairs:", len(train_pairs))
print("Eval pairs:", len(eval_pairs))
pair_df[["pair_id", "split", "chosen_id", "rejected_id"]].head(10)

## Step 4: Build feature tensors and inspect one example pair

The toy policy scores a response by taking the dot product between a weight vector and the response feature vector. This is much smaller than a language model, but it preserves the ranking logic that DPO needs.

In [ ]:
feature_matrix = {
    row["response_id"]: torch.tensor(row["features"], dtype=torch.float32)
    for row in response_rows
}

response_lookup = {row["response_id"]: row for row in response_rows}

def feature_frame_for_pair(pair_record):
    chosen_features = feature_matrix[pair_record["chosen_id"]].tolist()
    rejected_features = feature_matrix[pair_record["rejected_id"]].tolist()
    return pd.DataFrame(
        {
            "feature": feature_names,
            "chosen": chosen_features,
            "rejected": rejected_features,
            "chosen_minus_rejected": [round(c - r, 3) for c, r in zip(chosen_features, rejected_features)],
        }
    )

example_pair = train_pairs[0]
print("Example prompt:", example_pair["prompt"])
feature_frame_for_pair(example_pair)

## Step 5: Define the frozen SFT reference policy and the trainable policy

In real DPO, the reference model is usually the SFT checkpoint. The trainable policy starts from that same model and then learns stronger preference margins without a separate reward model.

In [ ]:
reference_weights = torch.tensor([1.05, 1.20, 1.45, 0.98, 0.82], dtype=torch.float32)
policy_weights = reference_weights.clone().detach().requires_grad_(True)

weight_df = pd.DataFrame(
    {
        "feature": feature_names,
        "reference_weight": reference_weights.tolist(),
        "initial_policy_weight": policy_weights.detach().tolist(),
    }
)

weight_df

## Step 6: Compute a single DPO loss by hand

The key idea is simple: compare the policy preference margin against the reference preference margin. If the policy does not prefer the chosen answer more strongly than the reference does, the loss stays high.

In [ ]:
def dpo_pair_loss(policy_vector, reference_vector, chosen_id, rejected_id, beta=0.7):
    chosen_tensor = feature_matrix[chosen_id]
    rejected_tensor = feature_matrix[rejected_id]

    policy_gap = torch.dot(policy_vector, chosen_tensor) - torch.dot(policy_vector, rejected_tensor)
    reference_gap = torch.dot(reference_vector, chosen_tensor) - torch.dot(reference_vector, rejected_tensor)
    scaled_margin = beta * (policy_gap - reference_gap)
    loss = -F.logsigmoid(scaled_margin)

    return loss, policy_gap, reference_gap, scaled_margin

demo_policy = reference_weights + torch.tensor([0.08, -0.04, 0.10, 0.05, 0.02], dtype=torch.float32)
demo_loss, demo_policy_gap, demo_reference_gap, demo_scaled_margin = dpo_pair_loss(
    demo_policy,
    reference_weights,
    example_pair["chosen_id"],
    example_pair["rejected_id"],
)

pd.DataFrame(
    [
        {"quantity": "policy_gap", "value": round(float(demo_policy_gap), 4)},
        {"quantity": "reference_gap", "value": round(float(demo_reference_gap), 4)},
        {"quantity": "beta_scaled_difference", "value": round(float(demo_scaled_margin), 4)},
        {"quantity": "dpo_loss", "value": round(float(demo_loss), 4)},
    ]
)

## Step 7: Measure the policy before DPO training

Because the initial policy matches the SFT reference, the first evaluation is the before snapshot. This gives us a clean baseline for later comparisons.

In [ ]:
def response_score(weight_vector, response_id):
    return float(torch.dot(weight_vector.detach(), feature_matrix[response_id]).item())

def evaluate_pairs(pair_records, weight_vector, split_name):
    rows = []
    for pair_record in pair_records:
        chosen_score = response_score(weight_vector, pair_record["chosen_id"])
        rejected_score = response_score(weight_vector, pair_record["rejected_id"])
        predicted = "chosen" if chosen_score >= rejected_score else "rejected"
        rows.append(
            {
                "split": split_name,
                "prompt_id": pair_record["prompt_id"],
                "prompt": pair_record["prompt"],
                "chosen_score": round(chosen_score, 4),
                "rejected_score": round(rejected_score, 4),
                "margin": round(chosen_score - rejected_score, 4),
                "correct": int(predicted == "chosen"),
            }
        )
    evaluation_df = pd.DataFrame(rows)
    summary = {
        "split": split_name,
        "pair_accuracy": round(float(evaluation_df["correct"].mean()), 4),
        "mean_margin": round(float(evaluation_df["margin"].mean()), 4),
    }
    return evaluation_df, summary

before_train_df, before_train_summary = evaluate_pairs(train_pairs, policy_weights, "train")
before_eval_df, before_eval_summary = evaluate_pairs(eval_pairs, policy_weights, "eval")

pd.DataFrame([before_train_summary, before_eval_summary])

## Step 8: Run a small toy DPO training loop

We update only the policy vector while keeping the reference frozen. Logged metrics show whether the policy is learning to prefer chosen responses more strongly than it did at the SFT stage.

In [ ]:
beta = 0.7
optimizer = torch.optim.Adam([policy_weights], lr=0.08)
history_rows = []

for epoch in range(1, 81):
    epoch_losses = []
    epoch_scaled_margins = []
    shuffled_pairs = train_pairs.copy()
    random.shuffle(shuffled_pairs)

    for pair_record in shuffled_pairs:
        loss, _, _, scaled_margin = dpo_pair_loss(
            policy_weights,
            reference_weights,
            pair_record["chosen_id"],
            pair_record["rejected_id"],
            beta=beta,
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(float(loss.detach()))
        epoch_scaled_margins.append(float(scaled_margin.detach()))

    train_eval_df, train_summary = evaluate_pairs(train_pairs, policy_weights, "train")
    eval_eval_df, eval_summary = evaluate_pairs(eval_pairs, policy_weights, "eval")

    history_rows.append(
        {
            "epoch": epoch,
            "loss": round(sum(epoch_losses) / len(epoch_losses), 6),
            "scaled_margin": round(sum(epoch_scaled_margins) / len(epoch_scaled_margins), 6),
            "train_pair_accuracy": train_summary["pair_accuracy"],
            "eval_pair_accuracy": eval_summary["pair_accuracy"],
            "train_mean_margin": train_summary["mean_margin"],
            "eval_mean_margin": eval_summary["mean_margin"],
        }
    )

history_df = pd.DataFrame(history_rows)
history_df.tail(10)

## Step 9: Plot the logged metrics

The loss should trend downward while pair accuracy and average chosen-minus-rejected margin move upward. Those are the core sanity checks for a DPO run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_df["epoch"], history_df["loss"], color="#4C72B0")
axes[0].set_title("DPO loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history_df["epoch"], history_df["train_pair_accuracy"], label="train", color="#55A868")
axes[1].plot(history_df["epoch"], history_df["eval_pair_accuracy"], label="eval", color="#C44E52")
axes[1].set_title("Pair accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["train_mean_margin"], label="train", color="#8172B2")
axes[2].plot(history_df["epoch"], history_df["eval_mean_margin"], label="eval", color="#CCB974")
axes[2].set_title("Chosen minus rejected margin")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Margin")
axes[2].legend()

plt.tight_layout()
plt.show()

## Step 10: Compare response rankings before and after DPO

A useful DPO notebook does not stop at scalar metrics. You also want to inspect which response wins before training and which one wins after training for the exact same prompts.

In [ ]:
after_train_df, after_train_summary = evaluate_pairs(train_pairs, policy_weights, "train")

comparison_rows = []
for pair_record in train_pairs:
    before_chosen_score = response_score(reference_weights, pair_record["chosen_id"])
    before_rejected_score = response_score(reference_weights, pair_record["rejected_id"])
    after_chosen_score = response_score(policy_weights, pair_record["chosen_id"])
    after_rejected_score = response_score(policy_weights, pair_record["rejected_id"])

    comparison_rows.append(
        {
            "prompt_id": pair_record["prompt_id"],
            "prompt": pair_record["prompt"],
            "before_winner": "chosen" if before_chosen_score >= before_rejected_score else "rejected",
            "after_winner": "chosen" if after_chosen_score >= after_rejected_score else "rejected",
            "before_margin": round(before_chosen_score - before_rejected_score, 4),
            "after_margin": round(after_chosen_score - after_rejected_score, 4),
            "chosen_preview": response_lookup[pair_record["chosen_id"]]["text"][:72] + "...",
            "rejected_preview": response_lookup[pair_record["rejected_id"]]["text"][:72] + "...",
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## Step 11: Evaluate on held-out prompts

Generalization matters. If the policy only improves the exact prompts it saw during training, the alignment stage is not doing much useful work.

In [ ]:
after_eval_df, after_eval_summary = evaluate_pairs(eval_pairs, policy_weights, "eval")

heldout_compare_df = before_eval_df.merge(
    after_eval_df[["prompt_id", "margin", "correct"]],
    on="prompt_id",
    suffixes=("_before", "_after"),
)

display(pd.DataFrame([before_eval_summary, after_eval_summary]))
heldout_compare_df

## Step 12: Inspect which features moved during DPO

Weight changes tell a simple story about what the policy learned to emphasize more strongly than the SFT reference. In this toy run, that usually means larger emphasis on safety, correctness, and specificity.

In [ ]:
weight_shift_df = pd.DataFrame(
    {
        "feature": feature_names,
        "reference_weight": reference_weights.tolist(),
        "dpo_weight": policy_weights.detach().tolist(),
    }
)
weight_shift_df["delta"] = weight_shift_df["dpo_weight"] - weight_shift_df["reference_weight"]
weight_shift_df["abs_delta"] = weight_shift_df["delta"].abs()

weight_shift_df.sort_values("abs_delta", ascending=False).reset_index(drop=True)

## Step 13: Save metrics and keep a DPO failure-mode checklist

A training notebook is more reusable when it exports both metrics and a short operational checklist. The checklist makes it easier to debug the next run when the numbers are worse than expected.

In [ ]:
metrics_summary = {
    "before_train": before_train_summary,
    "before_eval": before_eval_summary,
    "after_train": after_train_summary,
    "after_eval": after_eval_summary,
    "beta": beta,
}

history_path = ARTIFACT_DIR / "dpo_history.csv"
metrics_path = ARTIFACT_DIR / "dpo_metrics.json"

history_df.to_csv(history_path, index=False)
with metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics_summary, handle, indent=2)

dpo_failure_modes_df = pd.DataFrame(
    [
        {
            "failure_mode": "weak_or_noisy_pairs",
            "symptom": "Loss moves but pair accuracy stays flat.",
            "first_check": "Inspect score margins and annotation consistency.",
        },
        {
            "failure_mode": "bad_reference_checkpoint",
            "symptom": "The policy learns a strange style or overrefuses.",
            "first_check": "Verify that the frozen reference really is the intended SFT model.",
        },
        {
            "failure_mode": "beta_too_large",
            "symptom": "Margins explode and held-out behavior becomes brittle.",
            "first_check": "Reduce beta and inspect before and after examples.",
        },
        {
            "failure_mode": "prompt_leakage",
            "symptom": "Held-out gains look suspiciously perfect.",
            "first_check": "Audit prompt-level splits instead of row-level splits.",
        },
        {
            "failure_mode": "style_shortcuts",
            "symptom": "The policy learns to prefer longer or more templated answers regardless of quality.",
            "first_check": "Compare chosen and rejected lengths and formatting patterns.",
        },
    ]
)

print("Saved:", history_path)
print("Saved:", metrics_path)
dpo_failure_modes_df

## Key takeaways

- DPO starts from an SFT policy and compares it against a frozen SFT reference.
- The objective rewards stronger preference for chosen responses than the reference already provides.
- Logged metrics should be paired with before and after example inspection.
- Small local toy runs are enough to understand pair format, margins, and common failure modes before scaling to real models.